In [5]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from keras import Sequential, optimizers
from keras.callbacks import EarlyStopping
from keras.datasets import imdb
from keras.layers import (
    BatchNormalization,
    Dense,
    Dropout,
    Embedding,
    GlobalAveragePooling1D,
    Input,
)
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from tensorflow.keras.utils import pad_sequences


I0000 00:00:1788426872.115118   11665 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788426872.224283   11665 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788426874.334033   11665 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [25]:
document = "the cat sat on the map and the cat sat on the map"
tokens = document.split()
vocabulary = sorted(set(tokens))


In [ ]:
word_2_index = {word:index for index, word in enumerate(vocabulary)}

{'and': 0, 'cat': 1, 'map': 2, 'on': 3, 'sat': 4, 'the': 5}

In [ ]:
index_2_word = {index:word for word,index in word_2_index.items()}

{0: 'and', 1: 'cat', 2: 'map', 3: 'on', 4: 'sat', 5: 'the'}

In [45]:
windows_size = 2

X = []

y = []


for i in range(windows_size, len(tokens) - windows_size): # range(2, 11)
    
    context = tokens[i - windows_size: i] + tokens[i + 1: i + windows_size + 1]
    target = tokens[i]
    
    context_vectorize = np.zeros(len(vocabulary))
   
    for word in context:
        
        context_vectorize[word_2_index[word]] +=1
        
    context_vectorize /= len(context)
    
   
    
    target_vectorize = np.zeros(len(vocabulary))
    
    target_vectorize[word_2_index[target]] = 1
    
    
    X.append(context_vectorize)
    y.append(target_vectorize)
print(X[0])
print(y[0])

[0.   0.25 0.   0.25 0.   0.5 ]
[0. 0. 0. 0. 1. 0.]


In [2]:
SEED_NUM = 42
np.random.seed(SEED_NUM)
tf.random.set_seed(SEED_NUM)

In [3]:
(X_train ,y_train), (X_test, y_test) = imdb.load_data(num_words=10000)

In [4]:
print(X_test.shape)
print(y_test.shape)
print(X_train.shape)
print(y_train.shape)
print(X_train[0])
print(y_train[0])
print(y_test[0])
print(y_train[:10])
print(type(y_train))

(25000,)
(25000,)
(25000,)
(25000,)
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
1
0
[1 0 0 1 0 0 1 0 1 0

In [5]:
print(np.unique(y_train))
print(np.unique_counts(y_train))
labels, counts = np.unique_counts(y_train)
print(labels)
print(counts)

[0 1]
UniqueCountsResult(values=array([0, 1]), counts=array([12500, 12500]))
[0 1]
[12500 12500]


In [6]:
lengths = [len(document) for document in X_train]

print("Max: ",max(lengths))
print("Min: ",min(lengths))
print("Median: ",np.median(lengths))
print("percentile: ",np.percentile(lengths, [25, 50, 75, 90 , 95, 100]))
print("Mean: ", np.mean(lengths).round(2))

Max:  2494
Min:  11
Median:  178.0
percentile:  [ 130.  178.  291.  467.  610. 2494.]
Mean:  238.71


In [8]:
MAXLEN = 450

X_train = pad_sequences(X_train, maxlen = MAXLEN, padding = "post", truncating = "post") # shape = (10000, 450)
X_test = pad_sequences(X_test, maxlen = MAXLEN, padding = "post", truncating = "post") # shape = (10000, 450)
model = Sequential([
    
    Input(shape=(MAXLEN,), name = "input"),
    
    Embedding(input_dim = 10000, output_dim = 128, mask_zero = True),
    
    GlobalAveragePooling1D(),
    
    Dropout(0.25),
    Dense(units = 128, activation= "relu", name = "dense_one"),
    Dropout(0.3),
    Dense(units = 64, activation= "relu", name = "dense_two"),
    Dropout(0.4),
    Dense(units=1, activation="sigmoid"),
    
])

In [9]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 450, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_one (Dense)               │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_two (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,304,833 (4.98 MB)

 Trainable params: 1,304,833 (4.98 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics = ["accuracy"])

In [11]:
early_stopping = EarlyStopping(monitor = "val_loss", mode= "min", patience= 5, restore_best_weights = True, verbose =1 )

In [12]:
history = model.fit(X_train, y_train, validation_split = 0.2, epochs= 100, callbacks=early_stopping, batch_size=64, verbose = 1)

Epoch 1/100


I0000 00:00:1788378346.149345    3935 service.cc:153] XLA service 0x7bfbf4043cf0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788378346.149373    3935 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1788378346.199363    3935 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788378346.408202    3935 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1788378346.470767    3935 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1915__.18
I0000 00:00:1788378346.793592    3935 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set i

 13/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.5132 - loss: 0.6926

I0000 00:00:1788378350.555317    3935 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


311/313 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8079 - loss: 0.4202

I0000 00:00:1788378352.316548    3934 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1915__.18
I0000 00:00:1788378352.366023    3934 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1788378352.935921    3934 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8081 - loss: 0.4198

I0000 00:00:1788378354.942172    3933 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.8081 - loss: 0.4198 - val_accuracy: 0.8828 - val_loss: 0.2829
Epoch 2/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9127 - loss: 0.2287 - val_accuracy: 0.8902 - val_loss: 0.2871
Epoch 3/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9366 - loss: 0.1791 - val_accuracy: 0.8822 - val_loss: 0.3185
Epoch 4/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9414 - loss: 0.1567 - val_accuracy: 0.8816 - val_loss: 0.3507
Epoch 5/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9416 - loss: 0.1485 - val_accuracy: 0.8726 - val_loss: 0.3867
Epoch 6/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9340 - loss: 0.1534 - val_accuracy: 0.8792 - val_loss: 0.4099
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 1.


In [13]:
embedding_weights = model.get_layer("embedding").get_weights()[0]
print(embedding_weights.shape)

ValueError: No such layer: embedding. Existing layers are: ['embedding_1', 'global_average_pooling1d_1', 'dropout_3', 'dense_one', 'dropout_4', 'dense_two', 'dropout_5', 'dense_1'].

In [ ]:
pca = PCA(n_components=2)
embedding_2d = pca.fit_transform(embedding_weights)

In [ ]:
print(embedding_2d.shape)

In [ ]:
word_index = imdb.get_word_index()

In [ ]:
selected_words = [
    "excellent", "superb", "perfect", "brilliant", "amazing",
    "wonderful", "great", "best", "love", "beautiful",
    "worst", "terrible", "awful", "horrible", "waste",
    "bad", "boring", "stupid", "disappointing", "hate"
]

In [ ]:
for word in selected_words:
    token_id = word_index[word] + 3
    x, y = embedding_2d[token_id]
    print(word,": ", round(x,2), " - ",round(y, 2))

In [ ]:
positive_words = [
    "excellent", "superb", "perfect", "brilliant", "amazing",
    "wonderful", "great", "best", "love", "beautiful"
]

negative_words = [
    "worst", "terrible", "awful", "horrible", "waste",
    "bad", "boring", "stupid", "disappointing", "hate"
]

In [ ]:
plt.figure()

for word in positive_words:
    token_id = word_index[word] + 3
    x, y = embedding_2d[token_id]
    plt.scatter(x, y, color="blue", s=10, alpha= 0.7, label="Positive")
    plt.annotate(word, (x,y), xytext=(5,5), textcoords="offset points")
    
    
for word in negative_words:
    token_id = word_index[word]
    x, y = embedding_2d[token_id]
    plt.scatter(x, y, color="red", s=10, alpha= 0.7, label="Negative")
    plt.annotate(word, (x,y), xytext = (5,5), textcoords="offset points")
    
    
# plt.xlim(-0.20, 2.5)
# plt.ylim(-0.25, 0.1)

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("2D PCA Projection of Sentiment Word Embeddings")
plt.grid(True)
plt.tight_layout()
plt.show()